Below is the test cell to read all the Excel files

In [30]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "DDWEB_Downloads"
excel_files = sorted(DATA_DIR.glob("*.xlsx"))

print(f"Found {len(excel_files)} Excel files")
for f in excel_files:
    print(f.name)

Found 8 Excel files
DDweb_Auftrag_31032026_1706.xlsx
DDweb_Standort_01042026_1658.xlsx
DDweb_VI_Rohdaten_01042026_1726.xlsx
DDweb_VI_Rohdaten_01042026_1727.xlsx
DDweb_VI_Rohdaten_16102023_1722.xlsx
DDweb_VI_Rohdaten_31032026_1633.xlsx
DDweb_VI_Rohdaten_31032026_1642.xlsx
DDweb_VI_Rohdaten_31032026_1656.xlsx


In [31]:
df_Auftrag = pd.read_excel(excel_files[0])
df_Auftrag.head()

/Users/djiang/Berlin_traffic_data/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Startdatum,Enddatum,Beschreibung,Geräte-ID,Gerätetyp,Standorttitel,Stadt,Inhaber,Erstellt
0,2022-01-28 13:00:00,2023-04-29 11:00:00,Fahrtrichtung Nord,6429,DD.plus,Albanstraße Nr. 23 DD 6429,Berlin Tempelhof-Schöneberg,NaN,2022-02-28 13:23:57.410
1,2022-02-25 14:00:00,2023-03-18 08:00:00,Fahrtrichtung-Süd,5949,DD.plus,Bahnstraße Nr. 10 DD 5949,Berlin Tempelhof-Schöneberg,NaN,2022-02-28 13:26:39.890
2,2021-04-27 11:10:00,2049-01-01 00:00:00,i.H.-HNr. 4,5951,DD.plus,Boelkestraße Nr. 58 Fr-Ri. Süd DD 5951,Berlin -Tempelhof-Schöneberg,NaN,2023-07-28 15:41:38.778
3,2021-05-04 11:10:00,2049-01-01 00:00:00,Fahrtrichtung Nord,5950,DD.plus,Boelkestraße Nr. 65 Fr. Nord DD 5950,Berlin Tempelhof-Schöneberg,NaN,2021-05-05 10:12:33.113
4,2026-03-13 13:00:00,2049-01-01 00:00:00,DD 5949 Halker Zeile,5949,DD.plus,DD 5949 Halker Zeile,Berlin Tempelhof-Schöneberg,NaN,2026-03-17 09:44:25.166


In [32]:
df_Auftrag.describe()

,Startdatum,Enddatum,Geräte-ID,Inhaber,Erstellt
count,44,44,44.000000,0.0,44
mean,2023-10-01 18:47:46.204545,2058-07-05 15:40:48.818182,6879.500000,NaN,2023-10-11 16:53:09.306181
min,2020-03-18 10:39:00,2021-05-10 00:00:00,34.000000,NaN,2017-07-25 15:21:45
25%,2022-02-26 13:22:00,2025-01-25 11:00:00,6423.750000,NaN,2022-02-28 14:34:27.501500
50%,2023-04-22 13:00:00,2049-01-01 00:00:00,6429.000000,NaN,2023-06-12 23:07:35.251500
75%,2025-08-16 14:00:00,2100-01-01 23:59:59,7914.250000,NaN,2025-08-26 08:06:00.975250
max,2026-03-14 14:00:00,2100-01-01 23:59:59,8481.000000,NaN,2026-03-17 09:59:44.760000
std,NaN,NaN,1427.320314,NaN,NaN


In [33]:
# Ensure date columns are proper datetimes, not strings
df_Auftrag["Startdatum"] = pd.to_datetime(df_Auftrag["Startdatum"], errors="coerce")
df_Auftrag["Enddatum"]   = pd.to_datetime(df_Auftrag["Enddatum"],   errors="coerce")

#the Startdatum column — the date each deployment began. That range makes perfect sense: the oldest sensor was first deployed in 2017,
#and the most recently started deployment kicked off in March 2026.
#Sentinel dates used in Auftrag to mean "deployment still active"
ACTIVE_SENTINELS = {
    pd.Timestamp("2049-01-01"),
    pd.Timestamp("2100-01-01"),
}

df_Auftrag["is_active"] = df_Auftrag["Enddatum"].apply(
    lambda d: any(abs((d - s).days) < 2 for s in ACTIVE_SENTINELS) if pd.notna(d) else False
)
df_Auftrag["Enddatum_clean"] = df_Auftrag.apply(
    lambda row: pd.NaT if row["is_active"] else row["Enddatum"], axis=1
)

print(f"Auftrag rows: {len(df_Auftrag)}")
print(f"Unique device IDs in Auftrag: {df_Auftrag['Geräte-ID'].nunique()}")

Auftrag rows: 44
Unique device IDs in Auftrag: 30


In [34]:
df_Standort = pd.read_excel(excel_files[1])
df_Standort.head()
print(f"Standort rows: {len(df_Standort)}")

Standort rows: 45


---
## Drop Unimplemented Columns on Ingestion

Five columns are structurally zero across all files because the corresponding
sensor features were never activated. They are dropped immediately on load
to keep the working dataframe clean. The reasons are documented here:

| Column | Reason for dropping |
|---|---|
| `Schall (dB)` | Always 0 — sound measurement not implemented |
| `Abstand (cm)` | Always 0 — lateral distance not implemented |
| `Fahrspur` | Always 0 — lane indicator, not applicable on single-lane residential streets |
| `Geschwindigkeit (km/h)` | Always 0 — point speed not implemented; actual speed is in entry/exit columns |
| `Richtung` | Always 1 — one device per direction so this carries no information |

In [35]:
raw_files = sorted(DATA_DIR.glob("DDweb_VI_Rohdaten_*.xlsx"))
print(f"Found {len(raw_files)} raw data file(s):")
for f in raw_files:
    print(f"  {f.name}")

Found 6 raw data file(s):
  DDweb_VI_Rohdaten_01042026_1726.xlsx
  DDweb_VI_Rohdaten_01042026_1727.xlsx
  DDweb_VI_Rohdaten_16102023_1722.xlsx
  DDweb_VI_Rohdaten_31032026_1633.xlsx
  DDweb_VI_Rohdaten_31032026_1642.xlsx
  DDweb_VI_Rohdaten_31032026_1656.xlsx


In [36]:
COLS_TO_DROP = [
    "Schall (dB)",
    "Abstand (cm)",
    "Fahrspur",
    "Geschwindigkeit (km/h)",
    "Richtung",
]

RENAME_MAP = {
    "Geräte-ID":                          "device_id",
    "Datum":                              "datum_raw",
    "Eintrittsgeschwindigkeit (km/h)":    "speed_entry",
    "Austrittsgeschwindigkeit (km/h)":    "speed_exit",
    "Länge (dm)":                         "laenge_dm",
    "Klasse":                             "klasse",
    "Fahrzeugklassen-Bezeichnung":        "klasse_label",
}

frames = []
for fpath in raw_files:
    _df = pd.read_excel(fpath, dtype={"Geräte-ID": str})
    _df["source_file"] = fpath.name          # keep provenance
    _df = _df.drop(columns=COLS_TO_DROP, errors="ignore")
    _df = _df.rename(columns=RENAME_MAP)
    frames.append(_df)

df = pd.concat(frames, ignore_index=True)

print(f"Combined dataframe: {len(df):,} rows × {df.shape[1]} columns")
print(f"Columns kept: {list(df.columns)}")
df.head(3)

/Users/djiang/Berlin_traffic_data/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Combined dataframe: 134,070 rows × 8 columns
Columns kept: ['device_id', 'datum_raw', 'speed_entry', 'speed_exit', 'laenge_dm', 'klasse', 'klasse_label', 'source_file']


/Users/djiang/Berlin_traffic_data/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/djiang/Berlin_traffic_data/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,device_id,datum_raw,speed_entry,speed_exit,laenge_dm,klasse,klasse_label,source_file
0,5951,2026-03-25 00:19:54,62,70,42,7,Pkw,DDweb_VI_Rohdaten_01042026_1726.xlsx
1,5951,2026-03-25 00:21:52,49,51,38,7,Pkw,DDweb_VI_Rohdaten_01042026_1726.xlsx
2,5951,2026-03-25 00:51:22,37,43,38,7,Pkw,DDweb_VI_Rohdaten_01042026_1726.xlsx


---
## Timestamp Parsing with Multi-Format Detection

Two timestamp formats exist in the wild across export batches:
- **ISO format** (`YYYY-MM-DD HH:MM:SS`) — used in all recent exports; pandas reads this automatically
- **Legacy string format** (`DD/MM/YYYY HH:MM:SS`) — used in the 2022/2023 export; pandas will
  silently misparse this if you call `to_datetime` without specifying `dayfirst=True`

plan: detect the format per file and parse accordingly, then extract `datum` (date) and
`stunde` (hour 0–23) as separate columns for aggregation.

In [37]:
def parse_datum_column(series: pd.Series) -> pd.Series:
    """
    Parse a Datum column that may contain either:
    - datetime64 objects (already parsed by pandas on load from recent files)
    - strings in 'DD/MM/YYYY HH:MM:SS' format (legacy export format)
    Returns a datetime64 Series. Values that cannot be parsed become NaT.
    """
    if pd.api.types.is_datetime64_any_dtype(series):
        return series

    parsed = pd.to_datetime(series, dayfirst=True, errors="coerce") # It's a string column. Try the legacy DD/MM/YYYY format first (dayfirst=True)
    return parsed


# Parse each source file's dates separately
parsed_parts = []
for fpath in raw_files:
    mask = df["source_file"] == fpath.name
    parsed_parts.append(parse_datum_column(df.loc[mask, "datum_raw"]))

df["datum_parsed"] = pd.concat(parsed_parts).sort_index()

# For time components analysis
df["datum"]  = df["datum_parsed"].dt.date           # calendar date - for daily aggregation
df["stunde"] = df["datum_parsed"].dt.hour           # for hourly aggregation
df["wochentag"] = df["datum_parsed"].dt.day_name()  # for weekday patterns

# Flag any rows where parsing failed
df["flag_unparseable_timestamp"] = df["datum_parsed"].isna()
n_bad_ts = df["flag_unparseable_timestamp"].sum()

print(f"Timestamp parsing complete.")
print(f"  Rows with unparseable timestamp (flagged, not dropped): {n_bad_ts}")
print(f"  Date range: {df['datum_parsed'].min()} → {df['datum_parsed'].max()}")
df[["source_file", "datum_raw", "datum_parsed", "datum", "stunde", "wochentag"]].head(4)

Timestamp parsing complete.
  Rows with unparseable timestamp (flagged, not dropped): 0
  Date range: 2022-09-02 15:26:34 → 2026-03-31 23:57:06


,source_file,datum_raw,datum_parsed,datum,stunde,wochentag
0,DDweb_VI_Rohdaten_01042026_1726.xlsx,2026-03-25 00:19:54,2026-03-25 00:19:54,2026-03-25,0,Wednesday
1,DDweb_VI_Rohdaten_01042026_1726.xlsx,2026-03-25 00:21:52,2026-03-25 00:21:52,2026-03-25,0,Wednesday
2,DDweb_VI_Rohdaten_01042026_1726.xlsx,2026-03-25 00:51:22,2026-03-25 00:51:22,2026-03-25,0,Wednesday
3,DDweb_VI_Rohdaten_01042026_1726.xlsx,2026-03-25 01:25:16,2026-03-25 01:25:16,2026-03-25,1,Wednesday


---
## Validate Geräte-ID Against Reference Table

Every `Geräte-ID` in the raw data should appear in the Auftrag deployment table.
A mismatch means either (a) the sensor was deployed without being registered, or
(b) the georeferencing table has not been updated after a device was moved or replaced.
Flagged rows cannot be georeferenced and should be held back from spatial analysis.

In [38]:
# Build the set of all device IDs that have ever appeared in the Auftrag table.
known_device_ids = set(df_Auftrag["Geräte-ID"].astype(str).unique())

df["unknown_device_flagged"] = ~df["device_id"].astype(str).isin(known_device_ids)

unknown_ids = df.loc[df["unknown_device_flagged"], "device_id"].unique()
n_unknown_rows = df["unknown_device_flagged"].sum()

print(f"Known device IDs in Auftrag: {len(known_device_ids)}")
print(f"Unique device IDs in raw data: {df['device_id'].nunique()}")
print(f"Unregistered device IDs: {list(unknown_ids)}")
print(f"Rows flagged as unknown device (not dropped): {n_unknown_rows:,}")

Known device IDs in Auftrag: 30
Unique device IDs in raw data: 4
Unregistered device IDs: []
Rows flagged as unknown device (not dropped): 0


In [39]:
# Helpful summary: which devices appear in raw data, and whether they're known
device_summary = (
    df.groupby("device_id")
    .agg(row_count=("datum_parsed", "count"))
    .assign(in_auftrag=lambda x: x.index.isin(known_device_ids))
    .sort_values("row_count", ascending=False)
)
display(device_summary)

,row_count,in_auftrag
device_id,,
6426,54424,True
5950,41020,True
6429,36020,True
5951,2606,True


---
## Speed Plausibility Check

Speed is recorded as entry speed and exit speed. Flagged rows are logged but won't be dropped for this step. 


In [ ]:
# ── Speed plausibility thresholds (adjustable) ───────────────────────────────────────────
SPEED_MAX_MOTORISED  = 150   # km/h — flag any motorised vehicle above this
SPEED_MIN_MOTORISED  = 1     # km/h — flag any motorised vehicle below this
SPEED_MAX_BICYCLE    = 50    # km/h — flag any bicycle above this

# Klasse codes that are motorised (i.e. NOT bicycle or unknown)
MOTORISED_CLASSES = {2, 3, 5, 7, 8, 9, 10, 11, 64}
BICYCLE_CLASS     = 230

def speed_flag(speed_col, klasse_col):
    """
    It will return True for rows where the speed value is implausible given the vehicle class.
    Motorised vehicles: flag if speed < SPEED_MIN_MOTORISED or > SPEED_MAX_MOTORISED
    Bicycles:          flag if speed > SPEED_MAX_BICYCLE
    At the same time, zero exit speed is flagged — it represents an undefined read, not a stopped vehicle.
    """
    is_motorised = klasse_col.isin(MOTORISED_CLASSES)
    is_bicycle   = klasse_col == BICYCLE_CLASS

    flagged_motor   = is_motorised & (
        (speed_col > SPEED_MAX_MOTORISED) | (speed_col < SPEED_MIN_MOTORISED)
    )
    flagged_bicycle = is_bicycle & (speed_col > SPEED_MAX_BICYCLE)
    flag_zero_exit    = speed_col == 0

    return flagged_motor | flagged_bicycle | flag_zero_exit

In [41]:
df["speed_entry_flagged"] = speed_flag(df["speed_entry"], df["klasse"])
df["speed_exit_flagged"] = speed_flag(df["speed_exit"], df["klasse"])

# A row is flagged if either entry or exit speed is flagged
df["flag_speed"] = df["speed_entry_flagged"] | df["speed_exit_flagged"]

n_speed_entry = df["speed_entry_flagged"].sum()
n_speed_exit  = df["speed_exit_flagged"].sum()
n_speed_any   = df["flag_speed"].sum()

print(f"Rows with implausible entry speed: {n_speed_entry:,}")
print(f"Rows with implausible exit speed:  {n_speed_exit:,}")
print(f"Rows with implausible speed (either): {n_speed_any:,}")

Rows with implausible entry speed: 0
Rows with implausible exit speed:  1
Rows with implausible speed (either): 1


In [44]:
if n_speed_any > 0:
    print("\nSample flagged rows (investigate before the exclusion):")
    display(df[df["flag_speed"]][
        ["device_id", "datum_parsed", "klasse", "klasse_label", "speed_entry", "speed_exit", "source_file"]
        ].head(10)
        )


Sample flagged rows (investigate before the exclusion):


,device_id,datum_parsed,klasse,klasse_label,speed_entry,speed_exit,source_file
27742,5950,2026-03-29 09:02:25,11,Lfw,25,0,DDweb_VI_Rohdaten_01042026_1727.xlsx


---
## Duplicate Detection

A true duplicate is a row where essential/meaningful field is identical — same device,
same timestamp, same class, same speed, same length -> This might indicate a data transmission error 
(the device sent the same record twice).
Flagged but not dropped — review before removing.

In [47]:
DEDUP_COLS = [
    "device_id",
    "datum_parsed",   # timestamp to the second
    "klasse",
    "speed_entry",
    "speed_exit",
    "laenge_dm",
]

# keep=False marks ALL copies of a duplicate as True, so the full set of duplicated rows is visible for investigation.
df["flag_duplicate"] = df.duplicated(subset=DEDUP_COLS, keep=False)
n_dup = df["flag_duplicate"].sum()

print(f"Rows flagged as duplicates: {n_dup:,}")

if n_dup > 0:
    print("\nDuplicate groups (sorted by device and timestamp):")
    display(
        df[df["flag_duplicate"]]
        .sort_values(["device_id", "datum_parsed"])
        [["device_id", "datum_parsed", "klasse", "klasse_label", "speed_entry", "speed_exit", "laenge_dm", "source_file"]]
        .head(20)
    )

Rows flagged as duplicates: 120

Duplicate groups (sorted by device and timestamp):


,device_id,datum_parsed,klasse,klasse_label,speed_entry,speed_exit,laenge_dm,source_file
43799,6426,2022-09-02 16:08:36,230,Fahrrad,13,12,16,DDweb_VI_Rohdaten_16102023_1722.xlsx
43800,6426,2022-09-02 16:08:36,230,Fahrrad,13,12,16,DDweb_VI_Rohdaten_16102023_1722.xlsx
44234,6426,2022-09-02 18:00:04,230,Fahrrad,20,21,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
44235,6426,2022-09-02 18:00:04,230,Fahrrad,20,21,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
44591,6426,2022-09-02 19:50:56,230,Fahrrad,15,14,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
44592,6426,2022-09-02 19:50:56,230,Fahrrad,15,14,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
45997,6426,2022-09-03 16:31:46,230,Fahrrad,14,13,14,DDweb_VI_Rohdaten_16102023_1722.xlsx
45998,6426,2022-09-03 16:31:46,230,Fahrrad,14,13,14,DDweb_VI_Rohdaten_16102023_1722.xlsx
46792,6426,2022-09-07 16:21:40,230,Fahrrad,15,13,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
46793,6426,2022-09-07 16:21:40,230,Fahrrad,15,13,15,DDweb_VI_Rohdaten_16102023_1722.xlsx


---
## Entry/Exit Speed Delta Check

A large difference between entry and exit speed (even though both speeds are plausible) can indicate a sensor read error. A legitimate vehicle
passage should show similar entry and exit speeds on a residential street, meaning the speed should not change dramatically.

In [48]:
SPEED_DELTA_MAX      = 40    # km/h — flag entry/exit speed difference above this

df["speed_delta"] = (df["speed_entry"] - df["speed_exit"]).abs()

df["flag_speed_delta"] = df["speed_delta"] > SPEED_DELTA_MAX

n_delta = df["flag_speed_delta"].sum()

print(f"Rows where |entry − exit| > {SPEED_DELTA_MAX} km/h: {n_delta:,}")

if n_delta > 0:
    print("\nBreakdown by vehicle class:")
    delta_by_class = (
        df[df["flag_speed_delta"]]
        .groupby(["klasse", "klasse_label"])
        .agg(
            flagged_rows=("speed_delta", "count"),
            max_delta=("speed_delta", "max"),
            mean_delta=("speed_delta", "mean"),
        )
        .round(1)
        .sort_values("flagged_rows", ascending=False)
    )
    display(delta_by_class)

Rows where |entry − exit| > 40 km/h: 9

Breakdown by vehicle class:


,,flagged_rows,max_delta,mean_delta
klasse,klasse_label,,,
7,Pkw,4,101,58.25
10,Krad,3,57,51.333333
3,Lkw,1,43,43.0
11,Lfw,1,48,48.0


In [49]:
print("\nSample flagged rows:")
display(
    df[df["flag_speed_delta"]][
    ["device_id", "datum_parsed", "klasse", "klasse_label", "speed_entry", "speed_exit", "speed_delta"]
    ].sort_values("speed_delta", ascending=False).head(15)
)


Sample flagged rows:


,device_id,datum_parsed,klasse,klasse_label,speed_entry,speed_exit,speed_delta
43626,6426,2022-09-02 15:26:34,7,Pkw,111,10,101
1672,5951,2026-03-29 16:01:54,10,Krad,28,85,57
1489,5951,2026-03-29 00:33:12,10,Krad,29,83,54
62479,6426,2022-09-14 15:45:50,11,Lfw,18,66,48
8706,5950,2026-03-25 22:22:48,7,Pkw,13,59,46
27378,5950,2026-03-29 00:13:30,7,Pkw,26,71,45
29426,5950,2026-03-29 16:17:14,10,Krad,20,63,43
61871,6426,2022-09-14 09:36:34,3,Lkw,12,55,43
31061,5950,2026-03-30 01:25:08,7,Pkw,37,78,41
